In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!pip install "sentence-transformers<3.0" bert-score --quiet

import pandas as pd, numpy as np, torch, ast, json, gc, re
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from bert_score import score as bert_score_compute
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import sentence_transformers
print(f'sentence-transformers: {sentence_transformers.__version__}')
print(f'GPU disponibil: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

2026-06-02 20:25:38.818227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780431939.230855      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780431939.344801      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780431940.303545      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780431940.303588      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780431940.303591      58 computation_placer.cc:177] computation placer alr

sentence-transformers: 2.7.0
GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB


In [2]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f'Filme incarcate: {len(df):,}')
print(f'Coloane: {df.columns.tolist()}')

df['overview']         = df['overview'].fillna('').astype(str).str.strip()
df['tagline']          = df['tagline'].fillna('').astype(str).str.strip()
df['review_summary']   = df['review_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['overview_summary'] = df['overview_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['genres']           = df['genres'].fillna('').astype(str).str.strip()
df['keywords']         = df['keywords'].fillna('').astype(str).str.strip()
df['cast']             = df['cast'].fillna('').astype(str).str.strip()
df['director']         = df['director'].fillna('').astype(str).str.strip()

print()
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'keywords']:
    n   = (df[col] != '').sum()
    avg = df[col].str.split().str.len().mean()
    print(f'  {col:<22} acoperire: {n:>6,} ({n/len(df)*100:.1f}%)   medie: ~{avg:.0f} cuvinte')

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

  overview               acoperire: 40,197 (100.0%)   medie: ~47 cuvinte
  overview_summary       acoperire: 40,197 (100.0%)   medie: ~26 cuvinte
  review_summary         acoperire: 11,644 (29.0%)   medie: ~11 cuvinte
  tagline                acoperire: 40,197 (100.0%)   medie: ~9 cuvinte
  keywords               acoperire: 40,197 (100.0%)   medie: ~10 cuvinte


In [3]:
def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            names = [a['name'] for a in actors[:max_actors]
                     if isinstance(a, dict) and a.get('name')]
            return ', '.join(names)
    except Exception:
        pass
    return ''


def extract_keywords(kw_raw, max_kw=10):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except Exception:
        pass
    return ', '.join(w.strip() for w in kw_raw.split(',')[:max_kw])


def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]


def first_n_sentences(text, n=2):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return ' '.join(s for s in sentences[:n] if s)


def smart_overview(text, threshold=50, min_sent_words=15, fallback_words=60):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text  = text.strip()
    words = text.split()
    if len(words) <= threshold:
        return text
    first2 = first_n_sentences(text, n=2)
    if len(first2.split()) < min_sent_words:
        return ' '.join(words[:fallback_words])
    return first2

test_short = 'A toy cowboy feels threatened by a new spaceman toy.'
test_long  = 'In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Blackfoot woman as a way to gain entrance into her peoples rich lands, but finds she means more to him than a ticket to good beaver habitat. Their journey leads them deep into dangerous territory.'
test_abbr  = 'Lt. Col. Iceal Hambleton is a weapons countermeasures expert and when his aircraft is shot over Vietnam he must evade the enemy.'
print('smart_overview test:')
print(f'  scurt ({len(test_short.split())} cuv): "{smart_overview(test_short)}"')
print(f'  lung  ({len(test_long.split())} cuv):  "{smart_overview(test_long)}"')
print(f'  abbr  ({len(test_abbr.split())} cuv):  "{smart_overview(test_abbr)}"')
print('\nFunctii ajutatoare definite.')

smart_overview test:
  scurt (10 cuv): "A toy cowboy feels threatened by a new spaceman toy."
  lung  (61 cuv):  "In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Blackfoot woman as a way to gain entrance into her peoples rich lands, but finds she means more to him than a ticket to good beaver habitat."
  abbr  (22 cuv):  "Lt. Col. Iceal Hambleton is a weapons countermeasures expert and when his aircraft is shot over Vietnam he must evade the enemy."

Functii ajutatoare definite.


In [4]:
def build_doc_text_v7(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ov   = str(row.get('overview', '')).replace('nan', '').strip()
    plot = smart_overview(ov)
    if not plot:
        plot = ov
    if plot:
        parts.append('Plot: ' + plot)

    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev:
        parts.append('Critics: ' + rev[:200])

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 5)
    if kw:
        parts.append('Keywords: ' + kw)

    return ' '.join(parts)


def build_doc_text_no_review_v7(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ov   = str(row.get('overview', '')).replace('nan', '').strip()
    plot = smart_overview(ov)
    if not plot:
        plot = ov
    if plot:
        parts.append('Plot: ' + plot)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 5)
    if kw:
        parts.append('Keywords: ' + kw)

    return ' '.join(parts)


df['doc_text']           = df.apply(build_doc_text_v7, axis=1)
df['doc_text_no_review'] = df.apply(build_doc_text_no_review_v7, axis=1)

mask     = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f'Filme valide: {len(df_valid):,}')

real_leak = df_valid.apply(
    lambda r: len(str(r['tagline'])) > 15
              and str(r['tagline']).lower() in str(r['doc_text']).lower(),
    axis=1
).sum()
print(f'Leakage (tagline in doc_text): {real_leak} cazuri (~0.5% acceptabil)')

ex = df_valid[df_valid['title'] == 'Toy Story'].iloc[0]
print(f'\n--- Exemplu doc_text V7: {ex["title"]} ---')
print(ex['doc_text'])
print(f'\n--- [V6 folosea overview_summary] ---')
print(f'  V6 Plot: {ex["overview_summary"]}')
print(f'  V7 Plot: {smart_overview(ex["overview"])}')

Filme valide: 40,197
Leakage (tagline in doc_text): 190 cazuri (~0.5% acceptabil)

--- Exemplu doc_text V7: Toy Story ---
Toy Story. Plot: Led by Woody, Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Critics: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun. Genres: ['Family', 'Comedy', 'Animation', 'Adventure'] Keywords: rescue, friendship, mission, jealousy, villain

--- [V6 folosea overview_summary] ---
  V6 Plot: Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz.
  V7 Plot: Led by Woody, Andys toys live happily in hi

In [5]:
from transformers import AutoTokenizer

tok    = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
sample = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample]

with_rev = df_valid[df_valid['review_summary'] != '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] != '').sum()), random_state=42)
no_rev   = df_valid[df_valid['review_summary'] == '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] == '').sum()), random_state=42)

len_with = [len(tok(t, truncation=False)['input_ids']) for t in with_rev]
len_no   = [len(tok(t, truncation=False)['input_ids']) for t in no_rev]

print(f'doc_text V7 CU review   — Medie: {np.mean(len_with):.0f} | Max: {max(len_with)} | >256: {sum(l>256 for l in len_with)}/{len(len_with)}')
print(f'doc_text V7 FARA review — Medie: {np.mean(len_no):.0f}   | Max: {max(len_no)}   | >256: {sum(l>256 for l in len_no)}/{len(len_no)}')
print(f'Global                  — Medie: {np.mean(lengths):.0f}  | 99p: {np.percentile(lengths,99):.0f}   | >256: {sum(l>256 for l in lengths)}/{len(lengths)}')
print(f'\nNota: max_length=256 => batch=128 e fezabil')

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

doc_text V7 CU review   — Medie: 124 | Max: 175 | >256: 0/500
doc_text V7 FARA review — Medie: 72   | Max: 141   | >256: 0/500
Global                  — Medie: 87  | 99p: 156   | >256: 0/1000

Nota: max_length=256 => batch=128 e fezabil


In [6]:
BGE_QUERY_PREFIX = 'Represent this sentence: '

train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)
print(f'Training: {len(train_df):,} filme')
print(f'Eval:     {len(eval_df):,} filme')

train_examples = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

train_examples_no_review = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text_no_review']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

n_with = sum(1 for _, r in train_df.iterrows() if r['review_summary'] not in ('', 'nan'))
print(f'\nExemple principale:  {len(train_examples):,}')
print(f'  - cu review:       {n_with:,} ({n_with/len(train_df)*100:.1f}%)')
print(f'  - fara review:     {len(train_df)-n_with:,}')

print(f'\nExemplu pereche:')
ex = train_examples[0]
print(f'  Query: {ex.texts[0]}')
print(f'  Doc:   {ex.texts[1][:250]}...')

Training: 36,177 filme
Eval:     4,020 filme

Exemple principale:  35,788
  - cu review:       10,438 (28.9%)
  - fara review:     25,739

Exemplu pereche:
  Query: Represent this sentence: She loved and trusted him until he cut off her head.
  Doc:   Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolific, but now reclusive and alcoholic, movie star named Katharine Packard. While the rest of the house staff become suspicious of Vic...


In [7]:
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
OUTPUT_V7A = OUTPUT_PATH + 'sbert_v7a'
OUTPUT_V7B = OUTPUT_PATH + 'sbert_v7b'
device     = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

torch.cuda.empty_cache()
gc.collect()

sbert = SentenceTransformer(MODEL_NAME, device=str(device))

for mod in sbert.modules():
    if hasattr(mod, 'gradient_checkpointing_enable'):
        mod.gradient_checkpointing_enable()
        print('Gradient checkpointing: ON')
        break

print(f'Model: {MODEL_NAME}')
print(f'Device: {device}')
print(f'Embedding dim: {sbert.encode(["test"]).shape[1]}')

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU: {total_mem:.1f} GB total | {free_mem:.1f} GB libera')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gradient checkpointing: ON
Model: BAAI/bge-base-en-v1.5
Device: cuda:0
Embedding dim: 768
GPU: 14.6 GB total | 14.1 GB libera


In [8]:
torch.cuda.empty_cache()
gc.collect()

sbert  = sbert.to(device)
_tok   = sbert.tokenizer

def tokenize_batch(texts, max_length=256):
    return _tok(texts, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_loss_fn(emb_a, emb_p, scale=50.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE = 128
EPOCHS     = 3

train_dl     = DataLoader(train_examples, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=simple_collate)
total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f'Antrenare V7a: model={MODEL_NAME} | batch={BATCH_SIZE} | epoci={EPOCHS}')
print(f'  total steps={total_steps:,} | warmup={warmup_steps:,}')

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl),
                desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step, (anchors, positives) in pbar:
        enc_a = tokenize_batch(anchors)
        enc_p = tokenize_batch(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss_fn(emb_a, emb_p)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'Epoch {epoch+1}/{EPOCHS} — Loss mediu: {total_loss/len(train_dl):.4f}')
    sbert.save(OUTPUT_PATH + f'sbert_v7a_epoch{epoch+1}')

sbert.save(OUTPUT_V7A)
print(f'\nV7a salvat: {OUTPUT_V7A}')

Antrenare V7a: model=BAAI/bge-base-en-v1.5 | batch=128 | epoci=3
  total steps=840 | warmup=84


Epoch 1/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 2.6150


Epoch 2/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 2/3 — Loss mediu: 2.1586


Epoch 3/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 3/3 — Loss mediu: 1.9663

V7a salvat: /kaggle/working/sbert_v7a


In [9]:
BATCH_EMB = 256

queries_prefixed = [BGE_QUERY_PREFIX + t for t in df_valid['tagline'].tolist()]
docs             = df_valid['doc_text'].tolist()
docs_no_review   = df_valid['doc_text_no_review'].tolist()

sbert_base = SentenceTransformer(MODEL_NAME,  device=str(device))
sbert_v7a  = SentenceTransformer(OUTPUT_V7A,  device=str(device))

print('Generare embeddings Baseline BGE...')
q_emb_base = sbert_base.encode(queries_prefixed, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)
d_emb_base = sbert_base.encode(docs, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)

print('\nGenerare embeddings V7a (fine-tuned)...')
q_emb_v7a = sbert_v7a.encode(queries_prefixed, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)
d_emb_v7a = sbert_v7a.encode(docs, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)

print('\nGenerare embeddings V7a ablatie (fara review)...')
d_emb_v7a_norev = sbert_v7a.encode(docs_no_review, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_base.npy',      q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy',      d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_v7a.npy',       q_emb_v7a)
np.save(OUTPUT_PATH + 'd_emb_v7a.npy',       d_emb_v7a)
np.save(OUTPUT_PATH + 'd_emb_v7a_norev.npy', d_emb_v7a_norev)
print('\nEmbeddings salvate!')

Generare embeddings Baseline BGE...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings V7a (fine-tuned)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings V7a ablatie (fara review)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Embeddings salvate!


In [10]:
df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f'Genuri unice: {len(genre_to_indices)}')


def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                       batch_size=512, label=''):
    N    = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end    = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)
        top_k  = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k[i][np.argsort(scores[i][top_k[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 10000 == 0 and start > 0:
            print(f'  {label} {start}/{N}...')

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30, label=''):
    np.random.seed(42)
    indices    = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits       = {k: 0 for k in k_values}
    pool_sizes = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx)
        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr  = np.array(sorted(pool))
        pool_sizes.append(len(pool_arr))
        scores    = cosine_similarity(query_emb[idx:idx+1], doc_emb[pool_arr])[0]
        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f'  Pool mediu per query: {np.mean(pool_sizes):.0f} filme')
    return {k: hits[k] / n for k in k_values}


print('Functii de evaluare definite.')

Genuri unice: 19
Functii de evaluare definite.


In [11]:
results = {}

print('FULL POOL')

print('\nBaseline BGE (fara fine-tuning):')
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base, label='base')
print(f'  {results["base_full"]}')

print('\nV7a (BGE fine-tuned, smart_overview):')
results['v7a_full'] = evaluate_full_pool(q_emb_v7a, d_emb_v7a, label='v7a')
print(f'  {results["v7a_full"]}')

print('\nV7a Ablatie (fara review):')
results['v7a_norev_full'] = evaluate_full_pool(q_emb_v7a, d_emb_v7a_norev, label='v7a-norev')
print(f'  {results["v7a_norev_full"]}')

print('\nGENRE-FILTERED POOL')

print('\nBaseline BGE:')
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f'  {results["base_genre"]}')

print('\nV7a fine-tuned:')
results['v7a_genre'] = evaluate_genre_filtered(q_emb_v7a, d_emb_v7a)
print(f'  {results["v7a_genre"]}')

with open(OUTPUT_PATH + 'results_v7a.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V7a salvate!')

FULL POOL

Baseline BGE (fara fine-tuning):
  {1: 0.06637311242132497, 3: 0.09836554966788566, 5: 0.11359056646018359, 10: 0.13816951513794562, 20: 0.1691917307261736}

V7a (BGE fine-tuned, smart_overview):
  {1: 0.1106052690499291, 3: 0.16881856854989177, 5: 0.2009602706669652, 10: 0.2512376545513347, 20: 0.3097494837923228}

V7a Ablatie (fara review):
  {1: 0.11008284200313456, 3: 0.16610692340224395, 5: 0.196183794810558, 10: 0.245814364256039, 20: 0.3031818294897629}

GENRE-FILTERED POOL

Baseline BGE:
  Pool mediu per query: 15835 filme
  {1: 0.089, 3: 0.12333333333333334, 5: 0.14566666666666667, 10: 0.17433333333333334, 20: 0.208}

V7a fine-tuned:
  Pool mediu per query: 15835 filme
  {1: 0.12266666666666666, 3: 0.17933333333333334, 5: 0.20866666666666667, 10: 0.25, 20: 0.29933333333333334}

Rezultate V7a salvate!


In [12]:
HN_PER_QUERY = 6
MINE_TOPK    = 60

train_taglines_prefixed = [
    BGE_QUERY_PREFIX + str(row['tagline'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_docs = [
    str(row['doc_text'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_genres_list = [
    parse_genres(str(row['genres']))
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

print('Encoding embeddings pentru mining...')
q_emb_train = sbert_v7a.encode(
    train_taglines_prefixed, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)
d_emb_train = sbert_v7a.encode(
    train_docs, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)

TARGET_SAME  = HN_PER_QUERY // 2
TARGET_CROSS = HN_PER_QUERY - TARGET_SAME

print(f'\nMining Mixed HN: {TARGET_SAME} same-genre + {TARGET_CROSS} cross-genre per query')
print(f'  Top-{MINE_TOPK} candidati | {len(train_taglines_prefixed):,} query-uri')

hard_neg_examples = []

for i in tqdm(range(len(train_taglines_prefixed)), desc='Mining Mixed HN'):
    scores    = cosine_similarity(q_emb_train[i:i+1], d_emb_train)[0]
    scores[i] = -1.0
    top_idx   = np.argsort(scores)[::-1][:MINE_TOPK]

    genres_q   = set(train_genres_list[i])
    same_negs  = []
    cross_negs = []

    for idx in top_idx:
        genres_c = set(train_genres_list[idx])
        common   = len(genres_q & genres_c)

        if common >= 1 and len(same_negs) < TARGET_SAME:
            same_negs.append(train_docs[idx])
        elif common == 0 and len(cross_negs) < TARGET_CROSS:
            cross_negs.append(train_docs[idx])

        if len(same_negs) >= TARGET_SAME and len(cross_negs) >= TARGET_CROSS:
            break

    for hn in same_negs + cross_negs:
        hard_neg_examples.append(
            InputExample(texts=[
                train_taglines_prefixed[i],
                train_docs[i],
                hn
            ])
        )

print(f'\nHard negative triplete generate: {len(hard_neg_examples):,}')
if hard_neg_examples:
    ex = hard_neg_examples[0]
    print(f'\nExemplu triplet:')
    print(f'  Anchor:  {ex.texts[0]}')
    print(f'  Pozitiv: {ex.texts[1][:120]}...')
    print(f'  HardNeg: {ex.texts[2][:120]}...')

Encoding embeddings pentru mining...


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]


Mining Mixed HN: 3 same-genre + 3 cross-genre per query
  Top-60 candidati | 35,788 query-uri


Mining Mixed HN:   0%|          | 0/35788 [00:00<?, ?it/s]


Hard negative triplete generate: 200,223

Exemplu triplet:
  Anchor:  Represent this sentence: She loved and trusted him until he cut off her head.
  Pozitiv: Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolifi...
  HardNeg: You Go To My Head. Plot: Following a mysterious car accident in the desert, Dafne suffers from posttraumatic amnesia. Ja...


In [13]:
for var in ['sbert', 'sbert_base', 'sbert_v7a']:
    if var in globals():
        del globals()[var]
torch.cuda.empty_cache()
gc.collect()

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU libera dupa curatare: {free:.2f} GB')

sbert_v7b = SentenceTransformer(OUTPUT_V7A, device=str(device))
sbert_v7b = sbert_v7b.to(device)
_tok_v7b  = sbert_v7b.tokenizer

def tokenize_v7b(texts, max_length=256):
    return _tok_v7b(texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt')

def get_emb_v7b(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert_v7b(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_triplet_loss(emb_a, emb_p, emb_n, scale=50.0):
    emb_docs = torch.cat([emb_p, emb_n], dim=0)
    scores   = torch.mm(emb_a, emb_docs.T) * scale
    labels   = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def hn_collate(examples):
    return ([e.texts[0] for e in examples],
            [e.texts[1] for e in examples],
            [e.texts[2] for e in examples])

BATCH_HN  = 64
EPOCHS_HN = 2

hn_dl           = DataLoader(hard_neg_examples, batch_size=BATCH_HN,
                              shuffle=True, collate_fn=hn_collate)
total_steps_hn  = len(hn_dl) * EPOCHS_HN
warmup_steps_hn = int(0.05 * total_steps_hn)

optimizer_hn = AdamW(sbert_v7b.parameters(), lr=1e-5, weight_decay=0.01)
scheduler_hn = get_linear_schedule_with_warmup(optimizer_hn, warmup_steps_hn, total_steps_hn)
scaler_hn    = GradScaler('cuda')

print(f'Fine-tuning V7b cu Mixed Hard Negatives:')
print(f'  Start din: {OUTPUT_V7A}')
print(f'  batch={BATCH_HN} | triplete={len(hard_neg_examples):,} | epoci={EPOCHS_HN}')

for epoch in range(EPOCHS_HN):
    sbert_v7b.train()
    total_loss = 0.0
    optimizer_hn.zero_grad()

    pbar = tqdm(enumerate(hn_dl), total=len(hn_dl),
                desc=f'V7b Epoch {epoch+1}/{EPOCHS_HN}')
    for step, (anchors, positives, negatives) in pbar:
        enc_a = tokenize_v7b(anchors)
        enc_p = tokenize_v7b(positives)
        enc_n = tokenize_v7b(negatives)

        with autocast('cuda'):
            emb_a = get_emb_v7b(enc_a)
            emb_p = get_emb_v7b(enc_p)
            emb_n = get_emb_v7b(enc_n)
            loss  = mnr_triplet_loss(emb_a, emb_p, emb_n)

        scaler_hn.scale(loss).backward()
        scaler_hn.unscale_(optimizer_hn)
        torch.nn.utils.clip_grad_norm_(sbert_v7b.parameters(), max_norm=1.0)
        scaler_hn.step(optimizer_hn)
        scaler_hn.update()
        scheduler_hn.step()
        optimizer_hn.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'V7b Epoch {epoch+1} — Loss mediu: {total_loss/len(hn_dl):.4f}')

sbert_v7b.save(OUTPUT_V7B)
print(f'\nV7b salvat: {OUTPUT_V7B}')

GPU libera dupa curatare: 13.33 GB
Fine-tuning V7b cu Mixed Hard Negatives:
  Start din: /kaggle/working/sbert_v7a
  batch=64 | triplete=200,223 | epoci=2


V7b Epoch 1/2:   0%|          | 0/3129 [00:00<?, ?it/s]

V7b Epoch 1 — Loss mediu: 2.2207


V7b Epoch 2/2:   0%|          | 0/3129 [00:00<?, ?it/s]

V7b Epoch 2 — Loss mediu: 1.5841

V7b salvat: /kaggle/working/sbert_v7b


In [14]:
sbert_v7b_eval = SentenceTransformer(OUTPUT_V7B, device=str(device))

print('Generare embeddings V7b...')
q_emb_v7b = sbert_v7b_eval.encode(queries_prefixed, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)
d_emb_v7b = sbert_v7b_eval.encode(docs, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_v7b.npy', q_emb_v7b)
np.save(OUTPUT_PATH + 'd_emb_v7b.npy', d_emb_v7b)

print('\nEvaluare V7b full pool...')
results['v7b_full'] = evaluate_full_pool(q_emb_v7b, d_emb_v7b, label='v7b')
print(f'  {results["v7b_full"]}')

print('\nEvaluare V7b genre-filtered...')
results['v7b_genre'] = evaluate_genre_filtered(q_emb_v7b, d_emb_v7b)
print(f'  {results["v7b_genre"]}')

with open(OUTPUT_PATH + 'results_v7b.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V7b salvate!')

Generare embeddings V7b...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Evaluare V7b full pool...
  {1: 0.1888698161554345, 3: 0.29795755902181753, 5: 0.35400651789934573, 10: 0.43453491554096074, 20: 0.5208597656541533}

Evaluare V7b genre-filtered...
  Pool mediu per query: 15835 filme
  {1: 0.10133333333333333, 3: 0.157, 5: 0.17833333333333334, 10: 0.21466666666666667, 20: 0.24866666666666667}

Rezultate V7b salvate!


In [17]:
SAMPLE_BS  = 300
BATCH_BERT = 64

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return smart_overview(str(row['overview']))

refs      = [get_bert_text(i) for i in sample_idx]
hyps_base = [get_bert_text(get_top1_idx(q_emb_base, d_emb_base, i)) for i in sample_idx]
hyps_v7a  = [get_bert_text(get_top1_idx(q_emb_v7a,  d_emb_v7a,  i)) for i in sample_idx]
hyps_v7b  = [get_bert_text(get_top1_idx(q_emb_v7b,  d_emb_v7b,  i)) for i in sample_idx]

hit1_base = [1 if get_top1_idx(q_emb_base, d_emb_base, i) == i else 0 for i in sample_idx]
hit1_v7a  = [1 if get_top1_idx(q_emb_v7a,  d_emb_v7a,  i) == i else 0 for i in sample_idx]
hit1_v7b  = [1 if get_top1_idx(q_emb_v7b,  d_emb_v7b,  i) == i else 0 for i in sample_idx]

print(f'Sample BERTScore: {SAMPLE_BS} filme')
print(f'Hit@1 — base: {sum(hit1_base)/len(hit1_base):.3f} | v7a: {sum(hit1_v7a)/len(hit1_v7a):.3f} | v7b: {sum(hit1_v7b)/len(hit1_v7b):.3f}')

bert_results = {}
for name, hyps in [('base', hyps_base), ('v7a', hyps_v7a), ('v7b', hyps_v7b)]:
    print(f'\nBERTScore {name}...')
    _, _, F1 = bert_score_compute(
        hyps, refs, lang='en',
        model_type='bert-base-uncased',
        batch_size=BATCH_BERT, verbose=False
    )
    bert_results[name] = F1.numpy()
    hits_list = {'base': hit1_base, 'v7a': hit1_v7a, 'v7b': hit1_v7b}[name]
    f1_miss   = F1.numpy()[[h == 0 for h in hits_list]]
    print(f'  F1 global: {F1.mean():.4f} +- {F1.std():.4f}')
    if len(f1_miss) > 0:
        print(f'  F1 miss:   {f1_miss.mean():.4f}')

print('\nBERTScore calculat!')

Sample BERTScore: 300 filme
Hit@1 — base: 0.090 | v7a: 0.123 | v7b: 0.097

BERTScore base...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

  F1 global: 0.4872 +- 0.1690
  F1 miss:   0.4365

BERTScore v7a...
  F1 global: 0.5237 +- 0.1839
  F1 miss:   0.4566

BERTScore v7b...
  F1 global: 0.5109 +- 0.1658
  F1 miss:   0.4585

BERTScore calculat!


In [18]:
print(f'{"COMPARATIE MODELE — V6b vs V7":^80}')
print(f'{"Model":<44} {"Pool":<7} {"Hit@1":>6} {"Hit@5":>6} {"Hit@10":>7} {"Hit@20":>7}')

ref_vals = {
    '[Ref] V4b BGE + HN cross':         {1: 0.164, 5: 0.313, 10: 0.387, 20: 0.467},
    '[Ref] V4b genre-filtered':         {1: 0.100, 5: 0.166, 10: 0.198, 20: 0.240},
    '[Ref] V6b BGE + Keywords + MixedHN': {1: 0.193, 5: 0.360, 10: 0.443, 20: 0.529},
    '[Ref] V6b genre-filtered':         {1: 0.102, 5: 0.170, 10: 0.212, 20: 0.249},
}

rows_hk = [
    ('Baseline BGE (no fine-tune)',              'full',  'base_full'),
    ('[Ref] V4b BGE + HN cross',                'full',  None),
    ('[Ref] V6b BGE + Keywords + MixedHN',      'full',  None),
    ('V7a BGE + smart_overview (batch=128)',     'full',  'v7a_full'),
    ('V7a Ablatie (fara review)',                'full',  'v7a_norev_full'),
    ('V7b BGE + smart_overview + Mixed HN',     'full',  'v7b_full'),
    ('Baseline BGE',                            'genre', 'base_genre'),
    ('[Ref] V4b genre-filtered',                'genre', None),
    ('[Ref] V6b genre-filtered',                'genre', None),
    ('V7a fine-tuned',                          'genre', 'v7a_genre'),
    ('V7b + Mixed HN',                          'genre', 'v7b_genre'),
]

for label, pool, key in rows_hk:
    if key and key in results:
        r   = results[key]
        h1  = f"{r[1]*100:.1f}%"
        h5  = f"{r[5]*100:.1f}%"
        h10 = f"{r[10]*100:.1f}%"
        h20 = f"{r[20]*100:.1f}%"
    elif label in ref_vals:
        rv  = ref_vals[label]
        h1  = f"{rv[1]*100:.1f}%"
        h5  = f"{rv[5]*100:.1f}%"
        h10 = f"{rv[10]*100:.1f}%"
        h20 = f"{rv[20]*100:.1f}%"
    else:
        continue
    print(f'{label:<44} {pool:<7} {h1:>6} {h5:>6} {h10:>7} {h20:>7}')

print()
print(f'{"BERTSCORE F1 (smart_overview, N=300)":^65}')
print(f'{"Model":<30} {"BS-F1 global":>13} {"BS-F1 miss":>12}')
for name, hits_list, label in [
    ('base', hit1_base, 'Baseline BGE'),
    ('v7a',  hit1_v7a,  'V7a + smart_overview'),
    ('v7b',  hit1_v7b,  'V7b + smart_overview + Mixed HN'),
]:
    f1_all  = bert_results[name]
    f1_miss = bert_results[name][[h == 0 for h in hits_list]]
    print(f'{label:<30} {f1_all.mean():>13.4f} {f1_miss.mean():>12.4f}')

                         COMPARATIE MODELE — V6b vs V7                          
Model                                        Pool     Hit@1  Hit@5  Hit@10  Hit@20
Baseline BGE (no fine-tune)                  full      6.6%  11.4%   13.8%   16.9%
[Ref] V4b BGE + HN cross                     full     16.4%  31.3%   38.7%   46.7%
[Ref] V6b BGE + Keywords + MixedHN           full     19.3%  36.0%   44.3%   52.9%
V7a BGE + smart_overview (batch=128)         full     11.1%  20.1%   25.1%   31.0%
V7a Ablatie (fara review)                    full     11.0%  19.6%   24.6%   30.3%
V7b BGE + smart_overview + Mixed HN          full     18.9%  35.4%   43.5%   52.1%
Baseline BGE                                 genre     8.9%  14.6%   17.4%   20.8%
[Ref] V4b genre-filtered                     genre    10.0%  16.6%   19.8%   24.0%
[Ref] V6b genre-filtered                     genre    10.2%  17.0%   21.2%   24.9%
V7a fine-tuned                               genre    12.3%  20.9%   25.0%   29.9%
V7b + 